# Milestone 1: Exploratory Data Analysis

This notebook explores the Amazon Reviews 2023 Books dataset to inform retrieval model design decisions. It covers:

1. Dataset overview (fields, size, example records)
2. Inspection of sample records
3. Selection and justification of fields for retrieval
4. Text preprocessing decisions

**Data source:** McAuley-Lab/Amazon-Reviews-2023 via HuggingFace  
**Prerequisites:** Run `src/load_data.py` to generate the Parquet files.

## Setup

In [1]:
import re
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

DATA_DIR = Path("../data/processed/full")
reviews_path = DATA_DIR / "books_reviews.parquet"
metadata_path = DATA_DIR / "books_metadata.parquet"

# Open file handles — reads only the Parquet footer (schema + row group stats), not any row data.
reviews_file = pq.ParquetFile(reviews_path)
metadata_file = pq.ParquetFile(metadata_path)

print(f"Reviews:  {reviews_file.metadata.num_rows:,} rows x {reviews_file.metadata.num_columns} columns")
print(f"Metadata: {metadata_file.metadata.num_rows:,} rows x {metadata_file.metadata.num_columns} columns")

Reviews:  29,475,453 rows x 9 columns
Metadata: 4,448,181 rows x 14 columns


In [ ]:
# For exploration, we'll work with the sampled data, which is small enough to fit in memory.

SAMPLED_DIR = Path("../data/processed/sampled")
reviews_path_sampled = SAMPLED_DIR / "books_reviews_sample.parquet"
metadata_path_sampled = SAMPLED_DIR / "books_metadata_sample.parquet"

# If the sampled files don't exist, create them by running the `create_sample.py` script. This may take a few minutes. Updates will print to the console as the script runs, so you can monitor its progress. Once the sampled files are saved locally, you will not need to repeat this process.
if not reviews_path_sampled.exists() or not metadata_path_sampled.exists():
    import subprocess
    subprocess.run(["python", "src/create_sample.py"], cwd=Path(".."), check=True)

reviews_file_sampled = pq.ParquetFile(reviews_path_sampled)
metadata_file_sampled = pq.ParquetFile(metadata_path_sampled)

Sampling metadata...
  Full dataset: 4,448,181 rows
  20,000 books saved to data/processed/sampled/books_metadata_sample.parquet

Filtering reviews to sampled books...
  121,760 reviews saved to data/processed/sampled/books_reviews_sample.parquet


---
## 1. Dataset Overview

The dataset consists of two files:

- **books_reviews.parquet** — one row per user-book interaction, containing the review text, rating, and user/item identifiers.
- **books_metadata.parquet** — one row per book, containing structured item-level information such as title, description, price, and category.

The two files are linked by `parent_asin`, the canonical book identifier. A single book (`parent_asin`) may have multiple editions or formats (`asin`), so `parent_asin` is the appropriate join key.

In [3]:
# Schema is stored in the Parquet footer — no row data is read
print("=== Reviews schema ===")
for field in reviews_file.schema_arrow:
    print(f"  {field.name}: {field.type}")

=== Reviews schema ===
  rating: double
  title: string
  text: string
  asin: string
  parent_asin: string
  user_id: string
  timestamp: int64
  helpful_vote: int64
  verified_purchase: bool


In [4]:
# Load only numeric columns — skips text columns entirely
pq.read_table(reviews_path, columns=["rating", "helpful_vote", "timestamp"]).to_pandas().describe()

,rating,helpful_vote,timestamp
count,2.947545e+07,2.947545e+07,2.947545e+07
mean,4.414832e+00,1.777126e+00,1.457835e+12
std,1.066748e+00,1.538553e+01,1.413851e+11
min,0.000000e+00,-5.000000e+00,8.349701e+11
25%,4.000000e+00,0.000000e+00,1.390843e+12
50%,5.000000e+00,0.000000e+00,1.468269e+12
75%,5.000000e+00,1.000000e+00,1.558451e+12
max,5.000000e+00,1.813000e+04,1.694658e+12


In [5]:
print("=== Metadata schema ===")
for field in metadata_file.schema_arrow:
    print(f"  {field.name}: {field.type}")

=== Metadata schema ===
  main_category: string
  title: string
  average_rating: double
  rating_number: int64
  features: list<element: string>
  description: list<element: string>
  price: string
  store: string
  categories: list<element: string>
  details: string
  parent_asin: string
  bought_together: null
  subtitle: string
  author: string


In [6]:
# Load only numeric columns — skips text and list-type columns
pq.read_table(metadata_path, columns=["average_rating", "rating_number"]).to_pandas().describe()

,average_rating,rating_number
count,4.448181e+06,4.448181e+06
mean,4.405079e+00,4.444748e+02
std,6.243230e-01,3.655515e+03
min,1.000000e+00,1.000000e+00
25%,4.200000e+00,4.000000e+00
50%,4.500000e+00,1.300000e+01
75%,4.800000e+00,7.100000e+01
max,5.000000e+00,6.160400e+05


In [7]:
# Load only the rating column
ratings = pq.read_table(reviews_path, columns=["rating"]).to_pandas()["rating"]
print("=== Count of Reviews by Rating ===")
print(ratings.value_counts().sort_index().to_string())
print(f"\nMean rating: {ratings.mean():.2f}")

=== Count of Reviews by Rating ===
rating
0.0           4
1.0     1316085
2.0     1080897
3.0     2054057
4.0     4632932
5.0    20391478

Mean rating: 4.41


In [8]:
verified = pq.read_table(reviews_path, columns=["verified_purchase"]).to_pandas()["verified_purchase"]
total = len(verified)
counts = verified.value_counts()
labels = {True: "verified", False: "not verified"}
print("=== Count of Verified Purchases ===")
print(f"Verified: {counts[True]:,} ({counts[True] / total:.3%})")
print(f"Unverified: {counts[False]:,} ({counts[False] / total:.3%})")

=== Count of Verified Purchases ===
Verified: 20,566,389 (69.775%)
Unverified: 8,909,064 (30.225%)


In [9]:
# Load only the join key column from each file
review_asins = pq.read_table(reviews_path, columns=["parent_asin"]).to_pandas()["parent_asin"]
meta_asins = pq.read_table(metadata_path, columns=["parent_asin"]).to_pandas()["parent_asin"]

n_books_with_reviews = review_asins.nunique()
n_books_total = meta_asins.nunique()
avg_reviews = len(review_asins) / review_asins.nunique()
print("=== Number of Reviews per Book ===")
print(f"Number of books:              {n_books_total:,}")
print(f"Books with at least one review: {n_books_with_reviews:,}")
print(f"Percentage of books with reviews: {n_books_with_reviews / n_books_total:.3%}")
print(f"Average number of reviews per book: {avg_reviews:.1f}")

=== Number of Reviews per Book ===
Number of books:              4,448,181
Books with at least one review: 4,446,065
Percentage of books with reviews: 99.952%
Average number of reviews per book: 6.6


---
## 2. Inspection of Sample Records

Before any analysis, we inspect raw records to understand what values actually look like. We will use the sample of 20,000 books for this analysis, rather than the full dataset.

In [10]:
# Read only the first 5 rows using batch iteration
next(reviews_file_sampled.iter_batches(batch_size=5)).to_pandas()

,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Excellent traveler's resource!,I have been to the Tuscany/Florence region twi...,1118074661,1118074661,AFSKPY37N3C43SOI5IEXEK5JSIYA,1340230096000,0,False
1,5.0,A mouthwatering combination of memoirs and gre...,"Patty Pinner, author of the phenomenal [[ASIN:...",1561588482,1561588482,AFW2PDT3AMT4X3PYQG7FJZH5FXFA,1220790482000,2,False
2,5.0,So Much to Do!,This guide book is just about New York City. I...,0241368758,0241368758,AHV6QCNBJNSGLATP56JAWJ3C4G2A,1582766447655,0,False
3,5.0,Entertaining Biography of Novelist all Should ...,I recognized the name of novelist Barbara Pym ...,0008322201,0008322201,AEZ26WGWJ3EOQ4KWSHG77HJAG4EA,1663173665441,1,False
4,5.0,Excellent,What differentiates this from just being a ver...,0544749006,0544749006,AFJBKPK5W56XWSNPQU2WW66ISWYQ,1462166425000,12,False


In [11]:
# Read only the first 5 rows using batch iteration
next(metadata_file_sampled.iter_batches(batch_size=5)).to_pandas()

,main_category,title,average_rating,rating_number,features,description,price,store,categories,details,parent_asin,bought_together,subtitle,author
0,Books,"Mongoose, R.I.P.",4.5,47,"[Set during the 1962 Cuban missle crisis, this...","[From Publishers Weekly, Arguably, this is the...",4.59,William F. Buckley Jr. (Author),"[Books, Mystery, Thriller & Suspense, Thriller...","{""Publisher"": ""Random House; First Edition (No...",0394559312,None,"Hardcover – November 12, 1987",{'avatar': 'https://m.media-amazon.com/images/...
1,Books,"Condon's second island: A guidebook issue, 1971",5.0,1,[],[],None,Howard C Brooks (Author),[],"{""Publisher"": ""Geological Society of the Orego...",B0006YB27A,None,"Unknown Binding – January 1, 1971",None
2,Books,David Copperfield (Wordsworth Classics),4.6,627,[David Copperfield by Charles Dickens. Introdu...,"[Review, The most perfect of all the Dickens n...",3.95,Charles Dickens (Author),"[Books, Literature & Fiction, Classics]","{""Publisher"": ""Wordsworth Editions Ltd; Illust...",185326024X,None,"Paperback – Illustrated, August 4, 1997",None
3,Books,The Secrets of Life and Death: Answers For You...,4.8,20,[Is there life after death? Is there a rhyme o...,"[About the Author, Dr. Richard G. Shear Ed. D ...",14.99,Richard G. Shear Ed.D. (Author),"[Books, Religion & Spirituality, New Age & Spi...","{""Publisher"": ""CreateSpace Independent Publish...",1461086426,None,"Paperback – September 13, 2011",None
4,Books,Haunting of a Witch,4.5,610,[Her Troubled Past Hayden Wells deals with ple...,[],12.99,Suza Kates (Author),"[Books, Science Fiction & Fantasy, Fantasy]","{""Publisher"": ""Icasm Press (June 26, 2012)"", ""...",0984903054,None,"Paperback – June 26, 2012",{'avatar': 'https://m.media-amazon.com/images/...


In [12]:
# Null counts require reading every column, so this may be slow on the full dataset. Uncomment to run on sampled data.
print("=== Missing values: reviews ===")
print(pq.read_table(reviews_path_sampled).to_pandas().isna().sum().to_string())

=== Missing values: reviews ===
rating               0
title                0
text                 0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0


In [13]:
# Null counts require reading every column, so this may be slow on the full dataset. Uncomment to run on sampled data.
print("=== Missing values: metadata ===")
print(pq.read_table(metadata_path_sampled).to_pandas().isna().sum().to_string())

=== Missing values: metadata ===
main_category          1
title                  0
average_rating         0
rating_number          0
features               0
description            0
price                  0
store                783
categories             0
details                0
parent_asin            0
bought_together    20000
subtitle            2154
author              7125


In [14]:
# Load review text columns for length analysis
# The following code will take a while to run on the full dataset, since it needs to read all rows of the text columns. Consider running only on the sampled dataset for faster results.
reviews_text_cols = pq.read_table(reviews_path_sampled, columns=["title", "text"]).to_pandas()
review_title = reviews_text_cols["title"]
review_text = reviews_text_cols["text"]

text_lengths = review_text.dropna().str.len()
print("=== Review Length ===")
title_lengths = review_title.dropna().str.split().str.len()
print("Review title length (words):")
print(title_lengths.describe().drop("count").to_string())

word_counts = review_text.dropna().str.split().str.len()
print("Review text length (words):")
print(word_counts.describe().drop("count").to_string())
short = (word_counts < 5).sum()
print(f"Reviews with < 5 words: {short:,} ({short / len(word_counts):.1%})")

=== Review Length ===
Review title length (words):
mean     4.542387
std      3.493456
min      1.000000
25%      2.000000
50%      3.000000
75%      6.000000
max     64.000000
Review text length (words):
mean      78.661785
std      130.801780
min        0.000000
25%       14.000000
50%       34.000000
75%       86.000000
max     4420.000000
Reviews with < 5 words: 11,915 (9.8%)


In [15]:
# Fields used in retrieval (see part 3)
reviews_used = ["title", "text", "user_id", "parent_asin"]
metadata_used = ["title", "subtitle", "author", "description", "features", "categories", "store", "details", "average_rating", "parent_asin"]

reviews_df = pq.read_table(reviews_path_sampled, columns=reviews_used).to_pandas()
metadata_df = pq.read_table(metadata_path_sampled, columns=metadata_used).to_pandas()

print("=== Missing Values: Reviews (used fields) ===")
print(reviews_df.isna().sum().to_string())

print("=== Missing Values: Metadata (used fields) ===")
print(metadata_df.isna().sum().to_string())

=== Missing Values: Reviews (used fields) ===
title          0
text           0
user_id        0
parent_asin    0
=== Missing Values: Metadata (used fields) ===
title                0
subtitle          2154
author            7125
description          0
features             0
categories           0
store              783
details              0
average_rating       0
parent_asin          0


---
## 3. Field Selection for Retrieval

We are building both a BM25 retrieval model and a semantic (dense embedding) retrieval model. Below, we evaluate every field in both files and indicate whether which will be used and for what.

### Reviews fields (9 fields, after dropping `images` at load time)

| Field | Type | Use | Justification |
|---|---|---|---|
| `text` | str | Searching | Free-form review body |
| `title` | str | Searching | Short review headline |
| `rating` | float | Not used | Rating of book by reviewer |
| `helpful_vote` | int | Not used | Proxy for review quality |
| `verified_purchase` | bool | Not used | Proxy for review authenticity |
| `user_id` | str | Join key | Required to join against the `0core_timestamp_Books` train/valid/test splits |
| `parent_asin` | str | Join key | Canonical book identifier; used to join reviews to metadata |
| `asin` | str | Not used | Edition/format-level identifier; superseded by `parent_asin` for all joins |
| `timestamp` | int | Not used | Unix timestamp in milliseconds; temporal ordering is handled by the pre-built splits |

### Metadata fields (14 fields, after dropping `images` and `videos` at load time)

| Field | Type | Use | Justification |
|---|---|---|---|
| `title` | str | Searching | Book title; likely more useful for exact text matching than semantic |
| `description` | list[str] | Searching | Publisher description |
| `features` | list[str] | Searching | Bullet-point features |
| `categories` | list[str] | Searching | Genre and subject labels |
| `author` | str | Searching | Author name; use for author-based queries |
| `subtitle` | str | Searching | Secondary title |
| `store` | str | Searching | Publisher/store name; not useful for semantic matching, but keep for exact match on publisher name |
| `parent_asin` | str | Join key | Canonical book identifier; primary key for joining to reviews and splits |
| `main_category` | str | Not used | Top-level category (i.e. "Books"); uniform across this dataset so not useful |
| `average_rating` | float | Display | Aggregate rating signal; not a text term |
| `rating_number` | int | Not used | Review count |
| `price` | str | Not used | Stored as a string (inconsistent formatting) |
| `details` | str | Searching | Raw JSON string of product details (dimensions, ISBN, etc.); not useful for semantic matching but keep for exact match ISBN number |
| `bought_together` | list[str] | Not used | List of co-purchased ASINs |

### Selecting Columns

The goal of this system is to identify the best product matches using BM25 and semantic search. As such, we have elected to only incorporate text based columns that may be useful to exact matching or semantic search. We have excluded the text columns `bought_together` and `price` as they not relevant to semantic or BM25 searching. 

For the scope of Milestone 1, we have chosen to limit our algorithm to using pure BM25/semantic search to decide on the order of results. While we plan to present `average_rating` alongside each result, our algorithm will not be weighting the order of returned results by the average rating. If we were to expand on this system, we would incorporate non-text columns to add additional context and quality markings to each item. For example, we could use the `rating_number` column to weight the output of the semantic search by the number of reviews of each product, presenting highly reviewed products earlier than those with few reviews. Another example is that we could use the individual review ratings to determine the general sentiment of the review (reviews that rate the product highly are likely to express positive sentiments about the product), which could make our model's semantic understanding more accurate. Lastly, we could use `helpful_vote` or `verified_purchase` to weight the search results of individual reviews. These are interesting expansions that would likely provide value to our users, but are outside the scope of the current milestone.


### Per-book Document Contents

For both models, we will build a single document string per book by concatenating:

- `metadata.title`
- `metadata.subtitle`
- `metadata.author`
- `metadata.description` (joined if list)
- `metadata.features` (joined if list)
- `metadata.categories` (joined if list)
- Aggregated `reviews.title` and `reviews.text` (concatenation of reviews per book)

This concatenated string will be tokenized for BM25 and embedded for semantic search. The `average_rating` and `parent_asin` (unique ID) will be stored as metadata for each document.


---
## 4. Text Preprocessing Decisions

### Manual Inspection of Reviews

In [16]:
sample_reviews = pq.read_table(reviews_path_sampled, columns=["text"]).to_pandas()["text"].dropna().head(10)
for i, text in enumerate(sample_reviews, 1):
    print(f"--- Review {i} ---")
    print(text)
    print()

--- Review 1 ---
I have been to the Tuscany/Florence region twice in the last year and have another trip planned for this October. I originally wasn't going to select this book, thinking I already knew everything I needed to about this most beautiful area of Italy. But I received this Frommer's book and I am finding it a tremendous read! I am finding information on little known towns (for example Livorno) which I had never heard about in my travels (it's called the Venice of Tuscany). The book is a perfect size for travelers, recommending some wonderful restaurants in the cities (some of which I will also be trying in October)and many little known towns and tourist attractions (in addition to the more well known Florence/Pisa/Lucca/Sienna attractions. The pictures are beautiful, maps are all inclusive and the guide is written in clear English (to assist when in the areas where some may not speak the language). Hours of operation of tourist attractions as well as entrance fees are all i

### Planned preprocessing steps

BM25 operates on tokenized text. The preprocessing choices below are standard for BM25, and make sense after a manual inspection of raw text.

1. **Lowercasing** — BM25 is case-sensitive by default. We will implement lowercasing to prevent words like "Book" and "book" from being treated as distinct.
2. **Tokenization by whitespace and punctuation** — splits on spaces and strips characters like `.`, `,`, `!`, `?`, `"`. LangChain's `BM25Retriever` uses a default tokenizer that handles this.
3. **Stopword removal** — common English stopwords ("the", "a", "is") have high frequency across all documents and low discriminative value for BM25's IDF component. We will use NLTK's stopword list.
4. **No stemming/lemmatization** — Stemming and lemmatization are techniques that reduce a word to its root form. They are often used as a text preprocessing step, but we will not be using them because the dense retrieval model already handles semantic similarity.
5. **List field joining** — `description`, `features`, and `categories` are lists of strings; these will be joined with a space before tokenization. Since BM25 is a bag-of-words model, this is equivalent to treating each element independently. For semantic searching, while the model will consider token positions so the boundary between list elements could affect the embeddings, joining with a separator before embedding is standard practice and usually does not have a significant effect.
6. **Empty field handling** — missing or empty fields will be replaced with an empty string before concatenation, so the document is still valid.